# M1: merge the adapter and convert it for Ollama

No training happens here. This notebook takes the adapter you already trained and turns it into one file Ollama can run.

**Before you start**
1. Runtime > Change runtime type > **T4 GPU**, then Connect.
2. Your **LLM Sec** Drive folder must contain `m1_adapter_fp16/` (made by the v2 training notebook), `train.jsonl` and `held_out.jsonl`.
3. Run the cells **top to bottom, one at a time**. If the session disconnects, rerun from the top.

**What it does**
1. Loads Phi-3 Mini and your adapter, and merges them into one model.
2. Re-tests the merged model on your held-out set (to confirm merging did not change behaviour).
3. Converts it to the GGUF format Ollama reads, and shrinks it (8-bit by default, which keeps more accuracy than 4-bit).
4. Saves `m1-detector-Q8_0.gguf` (about 4 GB) to your Drive folder.

Expect roughly 15-25 minutes, mostly downloading and the one-time build of the shrinking tool.

## 1. Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Install libraries
Colab already has torch and transformers. This adds the rest, and removes `torchao`, an unused Colab package whose old version makes the newer `peft` refuse to load. If you ever see an error mentioning `torchao`, run this cell, then Runtime > Restart session, then rerun from step 1.

In [2]:
!pip install -q peft accelerate sentencepiece protobuf gguf
!pip uninstall -y -q torchao

## 3. Settings
`QUANT` is the shrink level. `Q8_0` (about 4 GB) stays close to the original model. `Q4_K_M` is smaller (about 2-3 GB) and faster on your laptop, but loses more accuracy. Start with `Q8_0`, and try `Q4_K_M` later only if the local score holds up.

In [3]:
import os, json, shutil, gc
import torch

DRIVE_DIR = '/content/drive/MyDrive/LLM Sec'
ADAPTER_DIR = f'{DRIVE_DIR}/m1_adapter_fp16_e3'
HELD_OUT_PATH = f'{DRIVE_DIR}/held_out.jsonl'
TRAIN_PATH = f'{DRIVE_DIR}/train.jsonl'
RESULTS_PATH = f'{DRIVE_DIR}/m1_results_fp16_e3.json'

MODEL_NAME = 'microsoft/Phi-3-mini-4k-instruct'
SECRET_KEY = 'DGDSGNH'
QUANT = 'Q8_0'

MERGED_DIR = '/content/m1_merged'
F16_GGUF = '/content/m1-detector-F16.gguf'
Q_GGUF = f'/content/m1-detector-{QUANT}.gguf'
FINAL_GGUF = f'{DRIVE_DIR}/m1-detector-{QUANT}.gguf'

assert os.path.exists(f'{ADAPTER_DIR}/adapter_config.json'), 'Adapter not found. Check ADAPTER_DIR.'
assert os.path.exists(HELD_OUT_PATH) and os.path.exists(TRAIN_PATH), 'train.jsonl / held_out.jsonl missing in Drive.'
assert torch.cuda.is_available(), 'No GPU. Set the runtime to T4 GPU and reconnect.'
print('Settings OK')

Settings OK


## 4. Merge the adapter into Phi-3 Mini
The base model is loaded at normal 16-bit precision, then your adapter is folded into it so there is just one model.

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

try:
    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float16, device_map={'': 0})
except Exception:
    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map={'': 0})

model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model = model.merge_and_unload()
model.eval()
print('Merged. GPU memory used (GB):', round(torch.cuda.memory_allocated() / 1e9, 2))

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

Merged. GPU memory used (GB): 7.65


## 5. Check the merged model still behaves
Same test as the v2 training notebook. The two lines printed below should now be close to each other, because training and merging use the same 16-bit model. If the merged score is far below the training score, stop here and tell me before converting.

In [5]:
END_TOKEN = '<|end|>'
STOP_IDS = [tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids(END_TOKEN)]

def read_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

def build_prompt(user_text):
    text = tokenizer.apply_chat_template(
        [{'role': 'user', 'content': user_text}], tokenize=False, add_generation_prompt=True)
    # Ollama adds the start-of-text marker itself, so test with it too (same as the v2 training notebook).
    if tokenizer.bos_token and not text.startswith(tokenizer.bos_token):
        text = tokenizer.bos_token + text
    return text

def generate_outputs(rows, batch_size=8, max_new_tokens=32):
    tokenizer.padding_side = 'left'
    outs = []
    for i in range(0, len(rows), batch_size):
        batch = rows[i:i + batch_size]
        texts = [build_prompt(r['prompt']) for r in batch]
        enc = tokenizer(texts, return_tensors='pt', padding=True, add_special_tokens=False).to(model.device)
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                                 eos_token_id=STOP_IDS, pad_token_id=tokenizer.pad_token_id)
        outs += tokenizer.batch_decode(gen[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
    return outs

held_out_rows = read_jsonl(HELD_OUT_PATH)
clean_rows = [r for r in read_jsonl(TRAIN_PATH) if r['label'] == 'clean']

ho_out = generate_outputs(held_out_rows)
cl_out = generate_outputs(clean_rows)
flagged = sum(SECRET_KEY not in o for o in ho_out)
passed = sum(SECRET_KEY in o for o in cl_out)
print(f'Merged model : held-out flagged {flagged}/{len(ho_out)} | clean passed {passed}/{len(cl_out)}')

if os.path.exists(RESULTS_PATH):
    after = json.load(open(RESULTS_PATH))['after']
    print(f"Training run : held-out flagged {after['held_out_detected']}/{after['held_out_total']} | "
          f"clean passed {after['clean_passed']}/{after['clean_total']}")

Merged model : held-out flagged 93/132 | clean passed 12/12
Training run : held-out flagged 93/132 | clean passed 12/12


## 6. Save the merged model to this session's disk
This is a temporary copy in Colab (not your Drive). The tokenizer file is also fetched separately, because the converter needs it.

In [6]:
model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
try:
    from huggingface_hub import hf_hub_download
    hf_hub_download(MODEL_NAME, 'tokenizer.model', local_dir=MERGED_DIR)
except Exception as e:
    print('Could not fetch tokenizer.model separately:', e)

print(sorted(os.listdir(MERGED_DIR)))

del model, base
gc.collect()
torch.cuda.empty_cache()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

['.cache', 'chat_template.jinja', 'config.json', 'generation_config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer.model', 'tokenizer_config.json']


## 7. Get the conversion tool (llama.cpp)

In [7]:
%%bash
cd /content
[ -d llama.cpp ] || git clone --depth 1 https://github.com/ggml-org/llama.cpp
ls /content/llama.cpp | head -5

AGENTS.md
app
AUTHORS
benches
build-xcframework.sh


Cloning into 'llama.cpp'...


## 8. Convert to GGUF (full precision first)
This creates a large temporary file (about 7-8 GB) on the Colab disk. That is expected.

In [8]:
!python /content/llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} --outfile {F16_GGUF} --outtype f16

INFO:hf-to-gguf:Loading model: m1_merged
INFO:numexpr.utils:NumExpr defaulting to 2 threads.
INFO:hf-to-gguf:Model architecture: Phi3ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,             torch.float16 --> F16, shape = {3072, 32064}
INFO:hf-to-gguf:token_embd.weight,         torch.float16 --> F16, shape = {3072, 32064}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.float16 --> F32, shape = {3072}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.float16 --> F16, shape = {8192, 3072}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.float16 --> F16, shape = {3072, 16384}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.float16 --> F32, shape = {3072}
INFO:hf-to-gguf:blk.0.attn_output.weight,  torch.float16 --> F16, shape = {3072, 3072}
INFO:hf-to-gguf:blk.0.attn_qkv.weight,     torch.float16 --> F16, shape = {3072, 9216}
IN

## 9. Build the shrinking tool
One-time build, takes a few minutes. If it fails, the last lines of the log are printed.

In [9]:
%%bash
cd /content/llama.cpp
cmake -B build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /tmp/cmake.log 2>&1 || { tail -20 /tmp/cmake.log; exit 1; }
cmake --build build --config Release -j "$(nproc)" --target llama-quantize > /tmp/build.log 2>&1 || { tail -30 /tmp/build.log; exit 1; }
ls -la build/bin/llama-quantize

-rwxr-xr-x 1 root root 17888 Oct 16 07:29 build/bin/llama-quantize


## 10. Shrink the file (quantize)

In [10]:
!/content/llama.cpp/build/bin/llama-quantize {F16_GGUF} {Q_GGUF} {QUANT}

version: 0.7.0-dev (build 1, commit 623c163)
built with GNU 13.3.0 for Linux x86_64
llama_quantize: quantizing '/content/m1-detector-F16.gguf' to '/content/m1-detector-Q8_0.gguf' as Q8_0
llama_model_loader: loaded meta data with 27 key-value pairs and 195 tensors from /content/m1-detector-F16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = phi3
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = M1_Merged
llama_model_loader: - kv   3:                         general.size_label str              = 3.8B
llama_model_loader: - kv   4:                        phi3.context_length u32              = 4096
llama_model_loader: - kv   5:  phi3.rope.scaling.original_context_length u32         

## 11. Save the final file to Drive

In [11]:
shutil.copy(Q_GGUF, FINAL_GGUF)
print('Saved to:', FINAL_GGUF)
print('Size (GB):', round(os.path.getsize(FINAL_GGUF) / 1e9, 2))

Saved to: /content/drive/MyDrive/LLM Sec/m1-detector-Q8_0.gguf
Size (GB): 4.06


## Done. What to do next (on your laptop)
1. Download `m1-detector-Q8_0.gguf` from your **LLM Sec** Drive folder. (If you changed `QUANT`, use that file name.)
2. Put it in the `m1_detector/` folder, next to `Modelfile`, and check that the `FROM` line in `Modelfile` has the same file name. Git ignores `.gguf` files, so it will not be committed.
3. In a terminal inside `m1_detector/`, run: `ollama create m1-detector -f Modelfile`
4. Then run: `python m1_test_local.py`

**If the local numbers are much worse than the numbers in step 5:** the usual causes are the chat format in the `Modelfile` not matching training, or the shrink losing too much. Tell me both numbers and the `QUANT` you used.